# Location Proxy Subgroup Analysis

This notebook offers a quick look at the postcode proxy experiments produced by the reusable pipeline. It focuses on subgroup behaviour (White vs Brown postcode variants) for each location prompt so you can sanity-check trends without leaving a notebook environment.


## Prerequisites

1. Export your API keys: `GEMINI_API_KEY` and `OPENAI_API_KEY`.
2. Run the reusable pipeline from the repository root, for example:
   ```bash
   python -m reusable_pipeline.cli --run-count 1 --prompt postcode_proxy --prompt location_cot_no_social
   ```
3. Confirm the pipeline created `outputs/combined/all_models_runs.csv`.

Once those steps are complete you can execute the cells below.


In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

# Resolve repository root so the notebook works whether launched from the root or notebooks/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').is_dir():
    parent = REPO_ROOT.parent
    if (parent / 'src').is_dir():
        REPO_ROOT = parent
    else:
        # Fall back to parent of parent if opened directly from notebooks/pipeline
        grandparent = parent.parent
        if (grandparent / 'src').is_dir():
            REPO_ROOT = grandparent
        else:
            raise FileNotFoundError('Could not locate repository root. Please run the notebook from the project root.')

OUTPUT_ROOT = REPO_ROOT / 'outputs'
MASTER_RUN_CSV = OUTPUT_ROOT / 'combined' / 'all_models_runs.csv'

LOCATION_SCENARIOS = [
    'postcode_proxy',
    'location_baseline_no_social',
    'location_instruction',
    'location_cot_no_social',
    'location_cot',
    'location_cot_engineered',
]

if not MASTER_RUN_CSV.exists():
    raise FileNotFoundError(f'Master run file not found at {MASTER_RUN_CSV}. Run the pipeline first.')


In [ ]:
master_df = pd.read_csv(MASTER_RUN_CSV)
master_df['decision'] = pd.to_numeric(master_df['decision'], errors='coerce')
master_df = master_df.dropna(subset=['decision'])
master_df['decision'] = master_df['decision'].astype(int)

location_df = master_df[master_df['scenario'].isin(LOCATION_SCENARIOS)].copy()
if location_df.empty:
    raise ValueError('No location scenarios found. Rerun the pipeline with location prompts enabled.')

print(f"Loaded {len(location_df):,} location records across {location_df['model'].nunique()} model(s).")
location_df.head()


In [ ]:
def compute_variant_metrics(df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df.groupby(['model', 'scenario', 'variant'])['decision']
        .mean()
        .reset_index(name='approval_rate')
    )
    grouped['approval_rate'] = grouped['approval_rate'].round(3)
    return grouped.sort_values(['model', 'scenario', 'variant']).reset_index(drop=True)

variant_metrics = compute_variant_metrics(location_df)
variant_metrics.head()


In [ ]:
def plot_variant_rates(df: pd.DataFrame, model: str, *, order=None) -> None:
    data = df[df['model'] == model]
    if data.empty:
        raise ValueError(f'No rows found for model {model}.')

    if order is None:
        order = [s for s in LOCATION_SCENARIOS if s in data['scenario'].unique()]

    plt.figure(figsize=(10, 4))
    sns.barplot(
        data=data,
        x='scenario',
        y='approval_rate',
        hue='variant',
        order=order,
    )
    plt.title(f'Approval rate by postcode variant ({model})')
    plt.ylabel('Approval rate')
    plt.xlabel('Scenario')
    plt.xticks(rotation=20, ha='right')
    plt.ylim(0, 1)
    plt.legend(title='Variant')
    plt.tight_layout()
    plt.show()

plot_variant_rates(variant_metrics, model='gemini-2.5-flash-lite')


### Inspect run-level details

If you need to dig deeper into individual runs, filter the master dataframe further. The helper below pulls a single run + scenario combination into a tidy view for manual inspection.


In [ ]:
def slice_run(df: pd.DataFrame, *, model: str, scenario: str, run: str) -> pd.DataFrame:
    subset = df[(df['model'] == model) & (df['scenario'] == scenario) & (df['run'] == run)].copy()
    if subset.empty:
        raise ValueError('No rows found for the provided filters.')
    columns = ['model', 'run', 'scenario', 'variant', 'applicant_id', 'applicant_uk_postcode', 'decision', 'justification', 'error']
    available = [col for col in columns if col in subset.columns]
    return subset[available].reset_index(drop=True)

slice_run(location_df, model='gemini-2.5-flash-lite', scenario='postcode_proxy', run='run_01').head()
